### Install new Libraries

In [1]:
# !pip install ddgs trafilatura -q
# !pip install openai-agents

### Setup

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS # duck duck go
import trafilatura
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [3]:
@function_tool
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [4]:
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
            return text
    print(f" \u274C Failed to fetch or extract text.")
    return f"Could not extract text from {url}. Try a different source."

### Step 2: The Agents

The Agents Prompts tells the LLM *who it is* and *how to behave*. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce

#### Research Agent

In [5]:
RESEARCH_AGENT_PROMPT = """
You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

***IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs look most relevant and why
- Which ones you will fetch and why
- Which ones you are skipping and why
Only AFTER writing out your reasoning should you call fetch_url.***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

In [6]:
research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web,fetch_url],
)

#### Orchestrator Agent

In [7]:
ORCHESTRATOR_AGENT_PROMPT = """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
"""

In [12]:
orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="gpt-6.1-terra", # reasoning
    tools=[research_agent.as_tool(tool_name="research_agent", tool_description="Research a topic and return a brief with key facts, statistics, themes and source URLs. Pass the topic as input.")]
)

### Let's run it!

In [16]:
result = await Runner.run(
    orchestrator_agent,
    input="Research the following topic and produce a comprehensive research brief: how will AI be used in Healthcare in 2040",
    max_turns=30
)

 ✅ search_web: Got results for AI in healthcare in 2040 clinical operational applications patient experience benefits limitations risks governance equity feasibility timelines
 ✅ fetch_url: Got 131 chars from https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/
 ✅ fetch_url: Got 131 chars from https://pmc.ncbi.nlm.nih.gov/articles/PMC12645148/
 ✅ search_web: Got results for artificial intelligence in healthcare future trends 2040
 ❌ Failed to fetch or extract text.
 ✅ fetch_url: Got 11564 chars from https://matter.health/posts/healthcare-2040-how-ai-will-tran
 ✅ search_web: Got results for AI in healthcare future trends prevention diagnosis treatment drug discovery care delivery administration barriers safety regulation unequal access
 ❌ Failed to fetch or extract text.
 ✅ fetch_url: Got 131 chars from https://pmc.ncbi.nlm.nih.gov/articles/PMC8285156/
 ✅ fetch_url: Got 131 chars from https://pmc.ncbi.nlm.nih.gov/articles/PMC12455834/
 ✅ search_web: Got results for AI in drug discovery fu

In [17]:
print(f"Agent: {result.last_agent.name}")
print(f"-----")
display(Markdown(result.final_output))

Agent: Orchestrator Agent
-----


The research agent reported a persistent technical issue with its search tool and did not return a research brief or source URLs. It indicated that any synthesis would need to distinguish established evidence from forecasts and acknowledge uncertainty, but it did not provide the synthesis.